# Control Variables - 15 Railways
11/07/2026, Kuba Kowalski 

Definition: 1 if a railway polyline intersects/crosses the province polygon, 0 otherwise. Placebo (railways that were proposed but never built) are EXCLUDED. 

In [8]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

ghana_rail_tab = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\15_railways\Replication_Files_Jedwab_Moradi\GIS_Files_for_Ghana\Railroads\Built\Railroad_Lines.TAB"
)

africa_rail_tab = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\15_railways\Replication_Files_Jedwab_Moradi\GIS_Files_for_Africa\Railroads\Built\Railroads.TAB"
)

json_out_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\15_railways\kuba_json_conversion"
)
json_out_dir.mkdir(parents=True, exist_ok=True)

combined_geojson = json_out_dir / "railways_jedwab_moradi_combined.geojson"

In [9]:
# ------------------------------------------------------------------
# LOAD RAILWAY DATA
# ------------------------------------------------------------------

ghana_rail = gpd.read_file(ghana_rail_tab)
africa_rail = gpd.read_file(africa_rail_tab)

print("Ghana rail CRS:", ghana_rail.crs)
print("Africa rail CRS:", africa_rail.crs)

print("Ghana rail geometry types:")
print(ghana_rail.geom_type.value_counts(dropna=False))

print("Africa rail geometry types:")
print(africa_rail.geom_type.value_counts(dropna=False))

# ------------------------------------------------------------------
# CRS HANDLING
# ------------------------------------------------------------------

if ghana_rail.crs is None:
    raise ValueError("Ghana railway CRS is missing. Define CRS manually before continuing.")

if africa_rail.crs is None:
    raise ValueError("Africa railway CRS is missing. Define CRS manually before continuing.")

# Use WGS84 for combined export
ghana_rail = ghana_rail.to_crs("EPSG:4326")
africa_rail = africa_rail.to_crs("EPSG:4326")

Ghana rail CRS: GEOGCS["unnamed",DATUM["MIF 0",SPHEROID["WGS 84 (MAPINFO Datum 0)",6378137.01,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Africa rail CRS: GEOGCS["unnamed",DATUM["MIF 0",SPHEROID["WGS 84 (MAPINFO Datum 0)",6378137.01,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Ghana rail geometry types:
MultiLineString    4
LineString         4
Name: count, dtype: int64
Africa rail geometry types:
LineString         137
MultiLineString     78
Name: count, dtype: int64


In [10]:
# ------------------------------------------------------------------
# STANDARDIZE AND COMBINE
# ------------------------------------------------------------------

ghana_rail["rail_source"] = "ghana_jedwab_moradi"
africa_rail["rail_source"] = "africa_jedwab_moradi"

# Keep all shared + non-shared columns safely
railways_combined = pd.concat(
    [ghana_rail, africa_rail],
    ignore_index=True,
    sort=False
)

railways_combined = gpd.GeoDataFrame(
    railways_combined,
    geometry="geometry",
    crs="EPSG:4326"
)

railways_combined["geometry"] = railways_combined.geometry.make_valid()

railways_combined = railways_combined[
    railways_combined.geometry.notna() &
    ~railways_combined.geometry.is_empty
].copy()

# Optional: keep only line geometries
railways_combined = railways_combined[
    railways_combined.geom_type.isin(["LineString", "MultiLineString"])
].copy()

# ------------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------------

railways_combined.to_file(
    combined_geojson,
    driver="GeoJSON"
)

print(f"\nSaved combined railway GeoJSON to:")
print(combined_geojson)

print("\nCombined railway features:", len(railways_combined))
print("Combined CRS:", railways_combined.crs)
print("Combined geometry types:")
print(railways_combined.geom_type.value_counts(dropna=False))


Saved combined railway GeoJSON to:
C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\15_railways\kuba_json_conversion\railways_jedwab_moradi_combined.geojson

Combined railway features: 223
Combined CRS: EPSG:4326
Combined geometry types:
LineString         141
MultiLineString     82
Name: count, dtype: int64


In [11]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

province_polygons = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"
)

railways_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\15_railways\kuba_json_conversion\railways_jedwab_moradi_combined.geojson"
)

out_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\15_railways"
)
out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

zone_id = "GEOLEVEL1"
control_var = "15_railway"

In [12]:
# ------------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------------

provinces = gpd.read_file(province_polygons)
railways = gpd.read_file(railways_file)

if provinces.crs is None:
    raise ValueError("Province file has no CRS.")

if railways.crs is None:
    raise ValueError("Railways file has no CRS.")

if zone_id not in provinces.columns:
    raise ValueError(f"Column '{zone_id}' not found in province file.")

# Normalize IDs
provinces[zone_id] = provinces[zone_id].astype(str).str.strip().str.zfill(6)

# Match CRS
if railways.crs != provinces.crs:
    railways = railways.to_crs(provinces.crs)

# Clean geometries
provinces["geometry"] = provinces.geometry.make_valid()
railways["geometry"] = railways.geometry.make_valid()

provinces = provinces[
    provinces.geometry.notna() &
    ~provinces.geometry.is_empty
].copy()

railways = railways[
    railways.geometry.notna() &
    ~railways.geometry.is_empty
].copy()

railways = railways[
    railways.geom_type.isin(["LineString", "MultiLineString"])
].copy()

# For province-level controls, compute once per GEOLEVEL1
provinces_unique = provinces.dissolve(
    by=zone_id,
    as_index=False
)

print(f"Province-cohort rows: {len(provinces)}")
print(f"Unique provinces: {len(provinces_unique)}")
print(f"Railway features: {len(railways)}")

Province-cohort rows: 2118
Unique provinces: 304
Railway features: 223


In [13]:
# ------------------------------------------------------------------
# SPATIAL JOIN: PROVINCES INTERSECTING RAILWAYS
# ------------------------------------------------------------------

intersections = gpd.sjoin(
    provinces_unique[[zone_id, "geometry"]],
    railways[["geometry"]],
    how="left",
    predicate="intersects"
)

railway_presence = (
    intersections
    .groupby(zone_id)["index_right"]
    .apply(lambda x: int(x.notna().any()))
    .reset_index(name=control_var)
)

# ------------------------------------------------------------------
# JOIN BACK TO ALL PROVINCE-COHORT ROWS
# ------------------------------------------------------------------

provinces_railway = provinces.merge(
    railway_presence,
    on=zone_id,
    how="left"
)

provinces_railway[control_var] = (
    provinces_railway[control_var]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------------
# EXPORT TABLE
# ------------------------------------------------------------------

railway_df = (
    provinces_railway[[zone_id, control_var]]
    .drop_duplicates(subset=[zone_id])
    .copy()
)

csv_path = out_dir / "15_railway_dummy.csv"
railway_df.to_csv(csv_path, index=False)

print(f"CSV saved to: {csv_path}")

# ------------------------------------------------------------------
# EXPORT SPATIAL FILE
# ------------------------------------------------------------------

gpkg_path = out_dir / "15_railway_dummy.gpkg"
provinces_railway.to_file(gpkg_path, driver="GPKG")

print(f"GPKG saved to: {gpkg_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nRailway dummy counts:")
print(railway_df[control_var].value_counts(dropna=False))

print("\nShare of provinces with railway:")
print(railway_df[control_var].mean())

print("Done.")

CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\15_railways\15_railway_dummy.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\15_railways\15_railway_dummy.gpkg

Railway dummy counts:
15_railway
1    153
0    151
Name: count, dtype: int64

Share of provinces with railway:
0.5032894736842105
Done.


In [14]:
# Joys of visualization - railway dummy

import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

africa_outline_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"

railways_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\15_railways\kuba_json_conversion\railways_jedwab_moradi_combined.geojson"
)

map_output_dir = out_dir / "maps"
map_output_dir.mkdir(exist_ok=True)

# ------------------------------------------------------------------
# LOAD OUTPUTS IF NEEDED
# ------------------------------------------------------------------

railway_gpkg = out_dir / "15_railway_dummy.gpkg"

if "provinces_railway" not in globals():
    provinces_railway = gpd.read_file(railway_gpkg)

africa = gpd.read_file(africa_outline_file)
railways = gpd.read_file(railways_file)

if africa.crs != provinces_railway.crs:
    africa = africa.to_crs(provinces_railway.crs)

if railways.crs != provinces_railway.crs:
    railways = railways.to_crs(provinces_railway.crs)

# ------------------------------------------------------------------
# MAP
# ------------------------------------------------------------------

var = "15_railway"

fig, ax = plt.subplots(figsize=(16, 20))

africa.plot(
    ax=ax,
    facecolor="none",
    edgecolor="lightgrey",
    linewidth=0.5
)

provinces_railway.plot(
    column=var,
    cmap="viridis",
    linewidth=0.4,
    edgecolor="black",
    legend=True,
    categorical=True,
    ax=ax
)

railways.plot(
    ax=ax,
    color="magenta",
    linewidth=1.0,
    alpha=0.8
)

ax.set_title("Railway dummy by province")
ax.set_axis_off()

minx, miny, maxx, maxy = africa.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

out_file = map_output_dir / "15_railway_dummy_map.png"

plt.savefig(
    out_file,
    dpi=800,
    bbox_inches="tight"
)

plt.close()

print(f"Saved: {out_file}")

Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\15_railways\maps\15_railway_dummy_map.png
